# ULB Credit Card Fraud — Card Transactions

| | |
|---|---|
| **Source** | Kaggle `mlg-ulb/creditcardfraud` |
| **Origin** | ULB Machine Learning Group / Worldline — real European card transactions, Sept 2013 |
| **License** | DbCL v1.0 (Database Contents License) |
| **Samples** | 284,807 transactions, 492 frauds (0.172%) |
| **Features** | `Time` (sec since first txn), `V1..V28` (PCA-anonymized), `Amount` |
| **Labels** | `Class` (1 = fraud) |
| **Caveats** | PCA anonymization → features not human-interpretable; 2-day window only. |
| **Production research suitability** | HIGH for fraud-model benchmarking; extreme class imbalance is realistic. |

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, iqr_outlier_share, save_clean, save_unified_part

D = RAW / "financial" / "creditcard"

In [2]:
df = pd.read_csv(D / "creditcard.csv")
print(df.shape)
df.head(3)

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


## Cleaning

In [3]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"dropped {before - len(df)} exact duplicates")
print("missing:", int(df.isna().sum().sum()))
assert df["Class"].isin([0, 1]).all()
df["Class"].value_counts()

dropped 1081 exact duplicates
missing: 0


Class
0    283253
1       473
Name: count, dtype: int64

## Timestamp normalization
`Time` = seconds since first transaction; capture began 2013-09-01 (per dataset
description: two days in September 2013). Anchor there — order-preserving, near-real.

In [4]:
anchor = pd.Timestamp("2013-09-01 00:00:00", tz="UTC")
df["event_time"] = anchor + pd.to_timedelta(df["Time"], unit="s")
df = df.sort_values("event_time").reset_index(drop=True)
df["event_time"].agg(["min", "max"])

min   2013-09-01 00:00:00+00:00
max   2013-09-02 23:59:52+00:00
Name: event_time, dtype: datetime64[us, UTC]

## Outliers + scaling
`Amount` heavy-tailed → keep raw + log1p. V-features already PCA-scaled.

In [5]:
print("Amount outlier share (IQR):", round(iqr_outlier_share(df["Amount"]), 4))
df["log1p_amount"] = np.log1p(df["Amount"])
df["hour_of_day"] = df["event_time"].dt.hour

Amount outlier share (IQR): 0.1117


## EDA

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["log1p_amount"], bins=60); axes[0].set_title("log1p(Amount)")
df.groupby("hour_of_day")["Class"].mean().plot(ax=axes[1], title="fraud rate by hour")
df["Class"].value_counts().plot.bar(ax=axes[2], title="class balance (log)"); axes[2].set_yscale("log")
plt.tight_layout(); plt.savefig("../reports/creditcard_eda.png", dpi=110); plt.show()

/tmp/ipykernel_179022/3671539822.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/creditcard_eda.png", dpi=110); plt.show()


In [7]:
numeric_summary(df, "creditcard").loc[["Time", "Amount", "V1", "V2", "V3"]]

,count,mean,std,min,25%,50%,75%,max
Time,283726.0,94811.077600,47481.047891,0.000000,54204.750000,84692.500000,139298.000000,172792.000000
Amount,283726.0,88.472687,250.399437,0.000000,5.600000,22.000000,77.510000,25691.160000
V1,283726.0,0.005917,1.948026,-56.407510,-0.915951,0.020384,1.316068,2.454930
V2,283726.0,-0.004135,1.646703,-72.715728,-0.600321,0.063949,0.800283,22.057729
V3,283726.0,0.001613,1.508682,-48.325589,-0.889682,0.179963,1.026960,9.382558


## Save clean + unified

In [8]:
save_clean(df, "creditcard")
dataset_report(df, "creditcard", label_col="Class",
               notes="PCA-anonymized; Time anchored to 2013-09-01 UTC (documented capture window).")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/creditcard.parquet (283,726 rows)


report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/creditcard_stats.json


{'dataset': 'creditcard',
 'rows': 283726,
 'columns': 34,
 'memory_mb': 76.0,
 'duplicate_rows': 0,
 'missing_by_column': {},
 'dtypes': {'Time': 'float64',
  'V1': 'float64',
  'V2': 'float64',
  'V3': 'float64',
  'V4': 'float64',
  'V5': 'float64',
  'V6': 'float64',
  'V7': 'float64',
  'V8': 'float64',
  'V9': 'float64',
  'V10': 'float64',
  'V11': 'float64',
  'V12': 'float64',
  'V13': 'float64',
  'V14': 'float64',
  'V15': 'float64',
  'V16': 'float64',
  'V17': 'float64',
  'V18': 'float64',
  'V19': 'float64',
  'V20': 'float64',
  'V21': 'float64',
  'V22': 'float64',
  'V23': 'float64',
  'V24': 'float64',
  'V25': 'float64',
  'V26': 'float64',
  'V27': 'float64',
  'V28': 'float64',
  'Amount': 'float64',
  'Class': 'int64',
  'event_time': 'datetime64[us, UTC]',
  'log1p_amount': 'float64',
  'hour_of_day': 'int32'},
 'notes': 'PCA-anonymized; Time anchored to 2013-09-01 UTC (documented capture window).',
 'label_distribution': {'0': 283253, '1': 473},
 'imbalance_rat

In [9]:
u = pd.DataFrame({
    "event_time": df["event_time"],
    "amount": df["Amount"],
    "severity": np.where(df["Class"] == 1, 3, 0).astype("int8"),
    "label": df["Class"].astype("Int8"),
    "time_is_synthetic": False,
})
attr_cols = [f"V{i}" for i in range(1, 29)]
u[attr_cols] = df[attr_cols].round(6)
u = to_unified(u, source_dataset="creditcard", event_domain="financial",
               event_type="card_txn", label_type="fraud", attributes_cols=attr_cols)
save_unified_part(u, "creditcard")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_creditcard.parquet (283,726 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,...,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,creditcard-0,2013-09-01 00:00:00+00:00,financial,card_txn,NaN,creditcard,<NA>,<NA>,<NA>,<NA>,...,149.62,NaN,NaN,NaN,0,0,fraud,<NA>,False,"{""V1"":-1.359807,""V2"":-0.072781,""V3"":2.536347,""..."
1,creditcard-1,2013-09-01 00:00:00+00:00,financial,card_txn,NaN,creditcard,<NA>,<NA>,<NA>,<NA>,...,2.69,NaN,NaN,NaN,0,0,fraud,<NA>,False,"{""V1"":1.191857,""V2"":0.266151,""V3"":0.16648,""V4""..."
2,creditcard-2,2013-09-01 00:00:01+00:00,financial,card_txn,NaN,creditcard,<NA>,<NA>,<NA>,<NA>,...,378.66,NaN,NaN,NaN,0,0,fraud,<NA>,False,"{""V1"":-1.358354,""V2"":-1.340163,""V3"":1.773209,""..."
